# Is the greenest composite a single day?

Examine pixel selection in a maximum-NDVI composite.

**Task** · Compare the composite NDVI map with each pixel’s observation date.

Run from top to bottom. Predict the result before changing code, then compare it with the output.

## Before you start

Use a CPU runtime. You need a registered Earth Engine Cloud project and permission to use it. See the [setup guide](https://github.com/steelpl/lecture/blob/main/CE776-climate-change-carbon-water-cycle/GEE_SETUP.md). Authenticate with your own account before querying data.

In [ ]:
%pip -q install earthengine-api==1.7.52 geemap==0.39.0

In [ ]:
import ee
import geemap

PROJECT_ID = ""  # Your Cloud project ID registered for Earth Engine
if not PROJECT_ID.strip():
    raise ValueError("Enter your registered Google Cloud project ID in PROJECT_ID")
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

In [ ]:
roi = ee.Geometry.Rectangle([126.80, 37.43, 127.15, 37.70], geodesic=False)

In [ ]:
def mask_s2(image):
    scl = image.select('SCL')
    clear = (scl.neq(1).And(scl.neq(3)).And(scl.neq(8))
             .And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11)))
    return (image.select(['B2', 'B3', 'B4', 'B8', 'B11'])
            .multiply(0.0001).updateMask(clear)
            .copyProperties(image, ['system:time_start']))
collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
              .filterBounds(roi).filterDate('2020-05-01', '2020-10-01')
              .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 60))
              .map(mask_s2))

def add_quality(image):
    ndvi = image.expression('(n-r)/(n+r)', {'n':image.select('B8'),'r':image.select('B4')}).rename('NDVI')
    day = ee.Date(image.get('system:time_start')).difference(ee.Date('2020-01-01'), 'day')
    date_band = ee.Image.constant(day).rename('day_since_jan1').toFloat()
    return image.addBands([ndvi, date_band])
quality = collection.map(add_quality)
greenest = quality.qualityMosaic('NDVI').clip(roi)
median_ndvi = quality.select('NDVI').median().clip(roi)

In [ ]:
print('scenes:', collection.size().getInfo())
print('NDVI mean:', greenest.select('NDVI').reduceRegion(ee.Reducer.mean(),roi,30,maxPixels=2_000_000).getInfo())

In [ ]:
Map = geemap.Map()
Map.centerObject(roi, 10)
Map.addLayer(greenest.select('NDVI'), {'min':0,'max':1,'palette':['white','green']}, 'Max NDVI')
Map.addLayer(median_ndvi, {'min':0,'max':1,'palette':['white','green']}, 'Median NDVI')
Map.addLayer(greenest.select('day_since_jan1'), {'min':121,'max':273,'palette':['blue','yellow','red']}, 'Selected day (0 = Jan 1)')
Map

Check whether all selected pixels have the same observation date. Explain the problem with interpreting this composite as one satellite acquisition. Inspect whether water or residual clouds could be selected because of unusually high NDVI.

**Takeaway** · NDVI is a vegetation-related index. It is not carbon uptake or evapotranspiration itself; those require additional models and validation.

## Data and interpretation

- [Sentinel-2 SR](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED)

Observation years define fixed examples, independent of the teaching year. The code demonstrates processing methods; inspect actual maps and values after authentication.